# Phase 3 — Baseline Models

Trains SARIMA and XGBoost, evaluates both on val and test splits,
and produces the benchmark table that Phase 4 foundation models must beat.

> **Pipeline upgrade (v2):** XGBoost now forecasts the **entire German grid mix** —
> 9 targets covering Load, Solar, Wind Onshore, Wind Offshore, Biomass, Run-of-River,
> Pumped Storage Generation, Other Renewables, and the dynamically calculated
> **`carbon_intensity_g_kwh`** for environmental footprint forecasting.
> All plotting cells below scale automatically to `len(TARGET_COLS)`.

**Rule**: any Phase 4 model that cannot beat XGBoost MAE is not worth deploying.


In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import logging

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

PROCESSED = Path('../data/processed')
RESULTS   = Path('../results')
MODELS    = Path('../models/baselines')
for d in [RESULTS, MODELS, RESULTS / 'baseline_plots']:
    d.mkdir(parents=True, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
print('Setup complete.')

## 0. Load feature dataset and split

In [ ]:
from src.features.pipeline import build_features, get_feature_cols, TARGET_COLS
from src.features.scaling import split_and_scale
from src.models import XGBoostForecaster, SARIMAForecaster, ResultsRegistry
from src.models.metrics import MetricResult

df = build_features(PROCESSED)
feat_cols = get_feature_cols(df)
print(f'Feature DataFrame: {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'Targets : {TARGET_COLS}')
print(f'Features: {len(feat_cols)} columns')

In [ ]:
splits = split_and_scale(
    df,
    target_cols=TARGET_COLS,
    feature_cols=feat_cols,
    train_end='2021-12-31',
    val_end='2022-12-31',
)
train, val, test = splits['train'], splits['val'], splits['test']
print(f'Train: {len(train):,}  ({train.index.min().date()} -> {train.index.max().date()})')
print(f'Val  : {len(val):,}  ({val.index.min().date()} -> {val.index.max().date()})')
print(f'Test : {len(test):,}  ({test.index.min().date()} -> {test.index.max().date()})')

## 1. XGBoost — fit all four targets

In [ ]:
HORIZON  = 24
registry = ResultsRegistry(RESULTS / 'baseline_metrics.jsonl')
xgb_models = {}

for target in TARGET_COLS:
    print(f'\n-- XGBoost: {target} --')
    model = XGBoostForecaster(target_col=target, horizon=HORIZON)
    model.fit(train, val_df=val)
    model.save(MODELS / f'xgboost_{target}')
    xgb_models[target] = model

    for split_name, split_df in [('val', val), ('test', test)]:
        result = model.evaluate(split_df, split_name=split_name)
        registry.add(result)

print('\nXGBoost fitting complete.')

## 2. Feature importance

In [ ]:
import math

n_targets = len(TARGET_COLS)
cols      = 2
rows      = math.ceil(n_targets / cols)
fig, axes = plt.subplots(rows, cols, figsize=(16, 5 * rows))
axes      = axes.flatten()

for i, target in enumerate(TARGET_COLS):
    imp = xgb_models[target].feature_importance(horizon_step=1, top_n=20)
    imp.plot.barh(ax=axes[i], color='steelblue')
    axes[i].set_title(f'{target} -- top 20 features (h=1)')
    axes[i].invert_yaxis()
    axes[i].set_xlabel('XGBoost gain')

# Hide unused subplot slots
for j in range(n_targets, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('XGBoost feature importances by target', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(RESULTS / 'baseline_plots' / 'xgb_importance_all.png', bbox_inches='tight')
plt.show()


## 3. Forecast vs actual (first 7 days of test set)

In [ ]:
fig, axes = plt.subplots(len(TARGET_COLS), 1,
                         figsize=(15, 3 * len(TARGET_COLS)), sharex=True)
if len(TARGET_COLS) == 1:
    axes = [axes]

N = 168

for ax, target in zip(axes, TARGET_COLS):
    preds  = xgb_models[target].predict(test)
    y_true = test[target].values
    ax.plot(y_true[:N],    lw=1.5, label='Actual',   color='steelblue')
    ax.plot(preds[:N, 0],  lw=1.2, label='XGB h=1',  color='crimson',    alpha=0.8)
    ax.plot(preds[:N, 23], lw=1.0, label='XGB h=24', color='darkorange', alpha=0.7, ls='--')
    ax.set_ylabel('value')
    ax.set_title(target)
    ax.legend(fontsize=8, loc='upper right')

axes[-1].set_xlabel('Hour (test set)')
plt.suptitle('XGBoost: h=1 vs h=24 forecasts, first 168 h of test set', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'baseline_plots' / 'xgb_forecast_vs_actual.png', bbox_inches='tight')
plt.show()


## 4. Horizon error curves

In [ ]:
import math

n_targets = len(TARGET_COLS)
cols      = 2
rows      = math.ceil(n_targets / cols)
fig, axes = plt.subplots(rows, cols, figsize=(14, 5 * rows))
axes      = axes.flatten()

for i, target in enumerate(TARGET_COLS):
    ax     = axes[i]
    preds  = xgb_models[target].predict(test)
    y_true = test[target].values
    maes, rmses = [], []

    for h in range(HORIZON):
        yp = preds[:, h]
        if h == 0:
            yt = y_true.copy()
        else:
            yt = np.full_like(y_true, np.nan)
            yt[:-h] = y_true[h:]
        mask = np.isfinite(yt) & np.isfinite(yp)
        maes.append(float(np.mean(np.abs(yt[mask] - yp[mask]))))
        rmses.append(float(np.sqrt(np.mean((yt[mask] - yp[mask]) ** 2))))

    hs = range(1, HORIZON + 1)
    ax.plot(hs, maes,  label='MAE',  color='steelblue', marker='o', ms=3)
    ax.plot(hs, rmses, label='RMSE', color='crimson',   marker='s', ms=3)
    ax.set_title(target)
    ax.set_xlabel('Horizon h (hours ahead)')
    ax.set_ylabel('Error')
    ax.legend(fontsize=9)
    ax.set_xticks([1, 6, 12, 18, 24])

# Hide unused subplot slots
for j in range(n_targets, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('XGBoost: MAE and RMSE by forecast horizon (test set)', fontsize=12)
plt.tight_layout()
plt.savefig(RESULTS / 'baseline_plots' / 'xgb_horizon_curves.png', bbox_inches='tight')
plt.show()


## 5. SARIMA — univariate baseline

Set `auto_order=False` to use a fixed order and skip the ~2 min `auto_arima` search.

In [ ]:
SARIMA_TARGETS = ['load_mw']
sarima_models  = {}

for target in SARIMA_TARGETS:
    print(f'Fitting SARIMA for {target} ...')
    sarima = SARIMAForecaster(
        target_col=target,
        horizon=HORIZON,
        auto_order=False,
    )
    sarima.fit(train)
    sarima.save(MODELS / f'sarima_{target}.pkl')
    sarima_models[target] = sarima

    for split_name, split_df in [('val', val), ('test', test)]:
        p      = sarima.predict_rolling(split_df)
        yt     = split_df[target].values
        result = MetricResult.from_arrays(
            model='sarima', target=target, split=split_name,
            horizon=HORIZON, y_true=yt, y_pred=p[:, 0],
        )
        registry.add(result)
        print(result)

print('\nSARIMA fitting complete.')

## 6. SARIMA vs XGBoost head-to-head on load_mw

In [ ]:
target       = 'load_mw'
N            = 168
xgb_preds    = xgb_models[target].predict(test)[:N, 0]
sarima_preds = sarima_models[target].predict_rolling(test)[:N, 0]
actual       = test[target].values[:N]

fig, ax = plt.subplots(figsize=(15, 4))
ax.plot(actual,       lw=2.0, label='Actual',  color='steelblue')
ax.plot(xgb_preds,    lw=1.5, label='XGBoost', color='crimson',    alpha=0.85)
ax.plot(sarima_preds, lw=1.3, label='SARIMA',  color='darkorange', alpha=0.8, ls='--')
ax.set_title(f'{target}: XGBoost vs SARIMA, first {N} h of test set')
ax.set_xlabel('Hour')
ax.set_ylabel('MW')
ax.legend()
plt.tight_layout()
plt.savefig(RESULTS / 'baseline_plots' / 'sarima_vs_xgb_load.png', bbox_inches='tight')
plt.show()

## 7. Benchmark table — numbers Phase 4 must beat

In [ ]:
results_df   = registry.to_dataframe()
test_results = results_df[results_df['split'] == 'test'].copy()

pivot = test_results.pivot_table(
    index='target',
    columns='model',
    values=['mae', 'rmse', 'mape', 'r2'],
    aggfunc='first',
).round(2)

print('\n=== BASELINE BENCHMARK TABLE (test set) ===')
print(pivot.to_string())
print('\nPhase 4 foundation models must beat the xgboost MAE column.')

pivot.to_csv(RESULTS / 'baseline_table.csv')
print(f'Saved -> {RESULTS / "baseline_table.csv"}')

## 8. Optional: Optuna HPO for XGBoost

In [ ]:
# Uncomment to run (~10 min, 50 trials)

# target = 'load_mw'
# best_params = xgb_models[target].tune(train, val, n_trials=50)
# print('Best params:', best_params)
#
# model_tuned = XGBoostForecaster(target_col=target, horizon=HORIZON, params=best_params)
# model_tuned.fit(train, val_df=val)
# result_tuned = model_tuned.evaluate(test, split_name='test')
# print(result_tuned)